# PRE-PROCESSING OF THE PNEUMONIA DATASET

During the eda, we notice two problems:
- Img do not have the same sizes
- The test folder is umbalanced and do not have enough img

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

## Resizing all img of each folder

In [18]:
def resize_img(folder, copy_folder):
    img_size = (256, 256)
    for label in os.listdir(folder):
        folder_path = os.path.join(folder, label)
        copy_folder_path = os.path.join(copy_folder, label)
        os.makedirs(copy_folder_path, exist_ok=True)
        for file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, file)
            copy_img_path = os.path.join(copy_folder_path, file)
            with Image.open(img_path) as img:
                img = img.convert("L")
                img = img.resize(img_size, Image.LANCZOS)
                img.save(copy_img_path)

train_dir = "../dataset_pneumonia/train"
val_dir = "../dataset_pneumonia/val"
test_dir = "../dataset_pneumonia/test"
pro_train_dir = "../dataset_resized/train"
pro_val_dir = "../dataset_resized/val"
pro_test_dir = "../dataset_resized/test"
resize_img(train_dir, pro_train_dir)
resize_img(val_dir, pro_val_dir)
resize_img(test_dir, pro_test_dir)


### Check if the resize works

In [ ]:
def check_size(folder):
    sizes = set()
    for label in os.listdir(folder):
        path = os.path.join(folder, label)
        for file in os.listdir(path):
            img_path = os.path.join(path, file)
            img = Image.open(img_path)
            sizes.add(img.size)
    print(sizes)

check_size(pro_train_dir)
check_size(pro_val_dir)
check_size(pro_test_dir)

{(256, 256)}
{(256, 256)}
{(256, 256)}


### Transform each img and their label into numpy arrays for model training

In [20]:
def transform_img(folder):
    images = []
    labels = []
    for label in os.listdir(folder):
        label_path = os.path.join(folder, label)
        for file in os.listdir(label_path):
            file_path = os.path.join(label_path, file)
            img = Image.open(file_path)
            img_array = np.array(img)
            images.append(img_array)
            labels.append(label)
    return np.array(images), np.array(labels)

### Affect features (img) and labels (NORMAL or PNEUMONIA)

In [ ]:
X_train, y_train = transform_img(pro_train_dir)
X_val, y_val = transform_img(pro_val_dir)
X_test, y_test = transform_img(pro_test_dir)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(6800, 256, 256)
(1700, 256, 256)
(624, 256, 256)


### Reshape features to only have one pixels

In [ ]:
h, w = X_train.shape[0], X_train.shape[1]

X_train = X_train.reshape(len(X_train), h * w)
X_val = X_val.reshape(len(X_val), h * w)
X_test = X_test.reshape(len(X_test), h * w)